# Chapter 1 · A tool's interface

An agent for Northside Veterinary Clinic. The agent and the model are always the same; what changes are the tools it receives. We start from the clinic's internal API as is and improve it in three steps: clear names and descriptions, results designed for the model and task-shaped tools. Each toolset is measured with twelve clinic tasks, checked by the answer and by the state of the schedule and of the notices.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Load the model and the clinic's data

The model takes about 8 GB; the first time it takes a few minutes to download. The data are nine made-up JSON files: owners, pets, medical history, vaccines, appointments, pharmacy, prices, invoices and vets.

In [ ]:
!pip install -q transformers==5.19.0 mcp==2.3.0  # the versions the book was checked with

In [ ]:
"""Chapter 1: a tool's interface. The same clinic tasks, with the same agent, and
four toolsets that get better step by step."""
import copy
import json
import re
import time
import unicodedata
import urllib.request
from datetime import date
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-1.7B"              # without a GPU: smaller, and a worse agent
MODEL = None                               # None: choose according to the hardware
DATA = Path("data")
REPOSITORY = "https://raw.githubusercontent.com/HernanDiaz/ai-tools-mcp-skills/main/en/"
TODAY = date(2026, 10, 20)                 # the date of the clinic's data


def load(name=None):
    """Loads the model and its tokenizer: on the GPU, in float16; if not, on the CPU."""
    gpu = torch.cuda.is_available()
    if name is None:
        name = GPU_MODEL if gpu else CPU_MODEL
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16 if gpu else torch.float32)
    model.to("cuda" if gpu else "cpu").eval()
    print(f"Model {name} on {'cuda' if gpu else 'cpu'}")
    return tok, model


def download(name):
    """A JSON file from the data folder; in Colab, it is downloaded."""
    path = DATA / name
    if not path.exists():
        DATA.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORY + "data/" + name, path)
    return json.loads(path.read_text(encoding="utf-8"))

In [ ]:
tok, model = load(MODEL)

In [ ]:
def reset():
    """Puts the clinic back as it was: the original schedule and no notices."""
    global APPOINTMENTS, NOTICES
    APPOINTMENTS, NOTICES = copy.deepcopy(APPOINTMENTS0), []


def strip_accents(text):
    """Lowercase and without diacritics, to compare names written in any form."""
    text = unicodedata.normalize("NFKD", str(text).lower())
    return "".join(c for c in text if not unicodedata.combining(c)).strip()


def load_data():
    """Loads the clinic's data and resets it."""
    global PETS, OWNERS, HISTORY, VACCINES, APPOINTMENTS0, PHARMACY, INVOICES
    global VETS, TIMES
    PETS, OWNERS = download("pets.json"), download("owners.json")
    HISTORY, VACCINES = download("history.json"), download("vaccines.json")
    APPOINTMENTS0, PHARMACY = download("appointments.json"), download("pharmacy.json")
    INVOICES, VETS = download("invoices.json"), download("vets.json")
    TIMES = [f"{h:02d}:{m:02d}" for h in range(9, 14) for m in (0, 30)] + \
            [f"{h:02d}:{m:02d}" for h in range(16, 19) for m in (0, 30)]
    reset()

In [ ]:
load_data()
print(len(OWNERS), "clients,", len(PETS), "pets and", len(APPOINTMENTS0),
      "appointments")
print(VETS)

## 2. The clinic's internal API

This is how the programmers of the web application wrote it. The three functions that act use the clinic's own (`book_appointment`, `cancel_appointment` and `notify`), which are the same for the four toolsets.

In [ ]:
def with_audit(record, n):
    """The control fields that the database adds to each record."""
    return {**record, "created_at": f"2026-0{1 + n % 9}-1{n % 10}T09:{10 + n % 50}:00Z",
            "updated_by": f"usr_{100 + n % 7}", "sync": True, "rev": n % 5 + 1}


def get_pat(pid):
    """Get patient."""
    if pid not in PETS:
        return {"status": 404}
    return {"status": 200, "id": pid, **with_audit(PETS[pid], 1),
            "history": [with_audit(v, i) for i, v in enumerate(HISTORY[pid])],
            "vaccines": [with_audit(v, i) for i, v in enumerate(VACCINES[pid])],
            "owner": {"id": PETS[pid]["owner"],
                      **with_audit(OWNERS[PETS[pid]["owner"]], 2)}}


def get_cli(cid):
    """Get client."""
    if cid not in OWNERS:
        return {"status": 404}
    pets = {p: d for p, d in PETS.items() if d["owner"] == cid}
    return {"status": 200, "id": cid, **with_audit(OWNERS[cid], 3),
            "pets": {p: with_audit(d, 4) for p, d in pets.items()},
            "invoices": [with_audit(f, i) for i, f in enumerate(INVOICES)
                         if f["owner"] == cid]}


def qry(table, field, value):
    """Query table."""
    rows = {"pets": PETS, "owners": OWNERS, "medicines": PHARMACY}.get(table)
    if rows is None:
        return {"status": 400}
    return {"status": 200, "rows": [
        {"id": i, **f} for i, f in rows.items()
        if strip_accents(value) in strip_accents(f.get(field, ""))]}


def sched_day(d):
    """Schedule."""
    return {"status": 200, "appointments": [with_audit(c, i)
                                            for i, c in enumerate(APPOINTMENTS)
                                            if c["date"] == d]}


def add_appt(pid, vid, d, t, rsn):
    """New appointment."""
    return book_appointment(pid, vid, d, t, rsn)


def del_appt(cid):
    """Delete appointment."""
    return cancel_appointment(cid)


def send_msg(cid, txt):
    """Send message."""
    return notify(cid, txt)


API_SET = {"get_pat": get_pat, "get_cli": get_cli, "qry": qry,
           "sched_day": sched_day, "add_appt": add_appt, "del_appt": del_appt,
           "send_msg": send_msg}

In [ ]:
def book_appointment(pet, vet, date, time, reason):
    """Books a free slot in the schedule; returns the appointment or the problem."""
    if pet not in PETS:
        return {"error": f"There is no pet {pet}."}
    if vet not in VETS:
        return {"error": f"There is no vet {vet}."}
    if any(c["date"] == date and c["time"] == time and c["vet"] == vet
           and c["status"] != "canceled" for c in APPOINTMENTS):
        return {"error": f"{VETS[vet]} already has an appointment on {date} "
                         f"at {time}."}
    appointment = {"id": f"AP-{len(APPOINTMENTS0) + len(APPOINTMENTS) + 1:04d}",
                   "date": date, "time": time, "vet": vet, "pet": pet,
                   "reason": reason, "status": "scheduled"}
    APPOINTMENTS.append(appointment)
    return appointment


def cancel_appointment(appointment):
    """Cancels a scheduled appointment."""
    for c in APPOINTMENTS:
        if c["id"] == appointment:
            if c["status"] != "scheduled":
                return {"error": f"Appointment {appointment} is \"{c['status']}\"."}
            c["status"] = "canceled"
            return {"canceled": appointment, "date": c["date"], "time": c["time"]}
    return {"error": f"There is no appointment {appointment}."}


def notify(owner, text):
    """Sends a notice to the owner (here it is stored in a list)."""
    if owner not in OWNERS:
        return {"error": f"There is no client {owner}."}
    NOTICES.append({"owner": owner, "text": text})
    return {"sent": True, "to": OWNERS[owner]["name"],
            "via": OWNERS[owner]["notify_by"]}

In [ ]:
print(get_pat("Lola"))
print(json.dumps(get_pat("P-012"), ensure_ascii=False)[:600], "…")
print(len(tok(json.dumps(get_pat("P-012"), ensure_ascii=False))["input_ids"]),
      "tokens")

## 3. The schemas and the agent

`automatic` generates the schemas from the signature and the docstring of each function, as many libraries do. The agent follows the ReAct pattern: a thought and a tool at each step.

In [ ]:
def tool(name, description, parameters):
    """Schema of a tool; the parameters without a default value are required."""
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object",
                       "required": [p for p, (_, req) in parameters.items() if req],
                       "properties": {p: {"type": "string", "description": d}
                                      for p, (d, _) in parameters.items()}}}}


def automatic(functions):
    """Schemas generated from the signature and the docstring, as many libraries do."""
    import inspect
    return [tool(n, (inspect.getdoc(f) or "").strip(),
                 {p: ("", True) for p in inspect.signature(f).parameters})
            for n, f in functions.items()]


SYSTEM = (
    "You are the assistant of the staff of Northside Veterinary Clinic, a veterinary "
    "clinic in Portland, Oregon. Today is Tuesday 2026-10-20. Do the task step by step "
    "using the tools. Never invent data that a tool can give you, and never say that "
    "an action was done unless a tool confirmed it. Call only one tool at a time, and "
    "before each tool call write one short line starting with 'Thought:' that says "
    "what you already know and what you need next. When the task is finished, answer "
    "in English, in one or two sentences.")

In [ ]:
for s in automatic(API_SET)[:3]:
    print(json.dumps(s, ensure_ascii=False))

In [ ]:
def generate(messages, schemas, max_tokens=400):
    """One answer of the model and its input and output tokens."""
    text = tok.apply_chat_template(messages, tools=schemas, tokenize=False,
                                   add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, pad_token_id=tok.eos_token_id)
    new = output[0, inputs["input_ids"].shape[1]:]
    return (tok.decode(new, skip_special_tokens=True).strip(),
            inputs["input_ids"].shape[1], len(new))


def read_calls(answer):
    """The calls the model asks for, written between <tool_call> and </tool_call>."""
    calls = []
    for block in re.findall(r"<tool_call>(.*?)</tool_call>", answer, re.S):
        try:
            call = json.loads(block)
            calls.append({"name": call["name"],
                          "arguments": call.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            calls.append({"name": None, "arguments": {}})
    return calls


def execute(call, functions):
    """Executes a call if the tool exists and the arguments fit."""
    function = functions.get(call["name"])
    if function is None:
        return {"error": f"There is no tool {call['name']}."}
    try:
        return function(**call["arguments"])
    except (TypeError, ValueError, KeyError) as e:
        return {"error": f"Invalid arguments: {e}"}


def agent(task, functions, schemas, max_steps=8, show=False):
    """Think, act, observe and repeat. Counts the tokens of the results."""
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": task}]
    trace = []
    for step in range(1, max_steps + 1):
        answer, n_in, n_out = generate(messages, schemas)
        calls = read_calls(answer)[:1]                   # one at a time
        thought = answer.split("<tool_call>")[0].strip()
        trace.append({"step": step, "input_tokens": n_in, "output_tokens": n_out,
                      "thought": thought, "calls": []})
        if not calls:
            trace[-1]["answer"] = answer
            if show:
                print(f"[{step}] Answer: {answer}")
            return answer, trace
        if show and thought:
            print(f"[{step}] {thought}")
        messages.append({"role": "assistant", "content": thought, "tool_calls": [
            {"type": "function", "function": c} for c in calls if c["name"]]})
        for call in calls:
            result = execute(call, functions)
            text = json.dumps(result, ensure_ascii=False)
            trace[-1]["calls"].append({**call, "result": result,
                                       "result_tokens": len(tok(text)["input_ids"])})
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": text})
            if show:
                args = json.dumps(call["arguments"], ensure_ascii=False)
                print(f"[{step}] {call['name']}({args})")
                print(f"     → {text[:150]}")
    return None, trace

In [ ]:
task = ("When is Lola, Martha Fisher's dog, due for her next rabies "
        "vaccine?")
reset()
answer, trace = agent(task, API_SET, automatic(API_SET), show=True)

## 4. Step 2: names and descriptions

The same functions with the same results; only what the model reads about them changes.

In [ ]:
def pet_record(pet_id):
    """The API's get_pat, with a clear name."""
    return get_pat(pet_id)


def client_record(client_id):
    """The API's get_cli, with a clear name."""
    return get_cli(client_id)


def search(table, field, value):
    """The API's qry, with a clear name."""
    return qry(table, field, value)


def schedule(date):
    """The API's sched_day, with a clear name."""
    return sched_day(date)


NAMES_SET = {"pet_record": pet_record, "client_record": client_record,
             "search": search, "schedule": schedule,
             "book_appointment": book_appointment,
             "cancel_appointment": cancel_appointment, "notify_owner": notify}


SCHEMAS_NAMES = [
    tool("pet_record", "Record of a pet of the clinic: details, owner, vaccines and "
         "medical history.", {"pet_id": ("Identifier, such as P-001.", True)}),
    tool("client_record", "Record of a client (owner): contact details, their pets "
         "and their invoices.", {"client_id": ("Identifier, such as C-001.", True)}),
    tool("search", "Searches for records whose field contains a value. Use it to find "
         "the identifier of a pet, an owner or a medicine from its name.",
         {"table": ("pets, owners or medicines.", True),
          "field": ("Field to search in, such as name.", True),
          "value": ("Text to search for.", True)}),
    tool("schedule", "Appointments of the clinic on a day.",
         {"date": ("Date in YYYY-MM-DD format.", True)}),
    tool("book_appointment", "Books an appointment in a free slot.",
         {"pet": ("Identifier of the pet, such as P-001.", True),
          "vet": ("Identifier of the vet: V1 (Dr. Anna Sutton), V2 (Dr. Paul Meyer) "
                  "or V3 (Dr. Lucy Fenton).", True),
          "date": ("YYYY-MM-DD.", True),
          "time": ("HH:MM, on the hour or half past.", True),
          "reason": ("Reason for the appointment.", True)}),
    tool("cancel_appointment", "Cancels a scheduled appointment.",
         {"appointment": ("Identifier of the appointment, such as AP-0026.", True)}),
    tool("notify_owner", "Sends a notice to the owner of a pet (email or SMS, as they "
         "prefer).", {"owner": ("Identifier of the client, such as C-001.", True),
                      "text": ("Text of the notice, in English.", True)}),
]

In [ ]:
reset()
answer, trace = agent(task, NAMES_SET, SCHEMAS_NAMES, show=True)

## 5. Step 3: results for the model

The same names and schemas; each tool returns only what helps to decide, with the math done and the errors explained.

In [ ]:
def age(birth_date):
    """Years completed on the date of the data."""
    n = date.fromisoformat(birth_date)
    return TODAY.year - n.year - ((TODAY.month, TODAY.day) < (n.month, n.day))


def pet_summary(pid):
    """The essentials of a pet, without the full medical history."""
    m = PETS[pid]
    return {"id": pid, "name": m["name"], "species": m["species"], "breed": m["breed"],
            "age_years": age(m["birth_date"]), "current_weight_kg": m["weight_kg"],
            "allergies": m["allergies"] or "none known",
            "owner": {"id": m["owner"], "name": OWNERS[m["owner"]]["name"]}}


def vaccines_with_status(pid):
    """Each vaccine with its next date and whether it is already overdue."""
    return [{"vaccine": v["vaccine"], "next": v["next"],
             "status": "overdue" if v["next"] < TODAY.isoformat() else "up to date"}
            for v in VACCINES[pid]]


def pet_record_3(pet_id):
    """The pet's record, summarized: no full history and no control fields."""
    if pet_id not in PETS:
        return {"error": f"There is no pet {pet_id}. Look it up by name with "
                         "search(table='pets', field='name', value=...)."}
    last = HISTORY[pet_id][-1] if HISTORY[pet_id] else None
    return {**pet_summary(pet_id), "vaccines": vaccines_with_status(pet_id),
            "last_visit": last and {k: last[k] for k in
                                    ("date", "reason", "treatment", "weight_kg")},
            "visits_in_history": len(HISTORY[pet_id])}


def client_record_3(client_id):
    """The client's record, summarized, with the total they owe."""
    if client_id not in OWNERS:
        return {"error": f"There is no client {client_id}. Look them up by name with "
                         "search(table='owners', field='name', value=...)."}
    unpaid = [f for f in INVOICES if f["owner"] == client_id and not f["paid"]]
    return {"id": client_id, "name": OWNERS[client_id]["name"],
            "notify_by": OWNERS[client_id]["notify_by"],
            "pets": [{"id": p, "name": d["name"], "species": d["species"]}
                     for p, d in PETS.items() if d["owner"] == client_id],
            "unpaid_invoices": [{"id": f["id"], "date": f["date"], "total": f["total"]}
                                for f in unpaid],
            "total_unpaid": round(sum(f["total"] for f in unpaid), 2)}


def search_3(table, field, value):
    """The search, with explained errors and short rows."""
    r = qry(table, field, value)
    if r["status"] != 200:
        return {"error": f"There is no table \"{table}\". The tables are pets, "
                         "owners and medicines."}
    if not r["rows"]:
        return {"error": f"No record in {table} has \"{value}\" in {field}.",
                "results": []}
    if table == "pets":
        return {"results": [pet_summary(f["id"]) for f in r["rows"]]}
    if table == "owners":
        return {"results": [{"id": f["id"], "name": f["name"]} for f in r["rows"]]}
    return {"results": [{"name": f["id"], "presentation": f["presentation"],
                         "dose_mg_per_kg": f["dose_mg_kg"], "stock": f["stock"],
                         "restock": f.get("restock", "no date")}
                        for f in r["rows"]]}


def schedule_3(date):
    """The day's appointments, one per line, and each vet's free times."""
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(date)):
        return {"error": f"The date \"{date}\" is not valid; use YYYY-MM-DD."}
    appts = [c for c in APPOINTMENTS if c["date"] == date and c["status"] != "canceled"]
    return {"date": date, "appointments": [f"{c['id']} {c['time']} {c['vet']} "
                                           f"{PETS[c['pet']]['name']} ({c['pet']})"
                                           for c in appts],
            "vets": VETS,
            "free_times": {v: [t for t in TIMES if not any(
                c["time"] == t and c["vet"] == v for c in appts)]
                for v in VETS}}


RESULTS_SET = {"pet_record": pet_record_3, "client_record": client_record_3,
               "search": search_3, "schedule": schedule_3,
               "book_appointment": book_appointment,
               "cancel_appointment": cancel_appointment, "notify_owner": notify}


SCHEMAS_RESULTS = SCHEMAS_NAMES                 # same schemas, other results

In [ ]:
n = lambda r: len(tok(json.dumps(r, ensure_ascii=False))["input_ids"])
print("Zeus's record:", n(get_pat("P-012")), "tokens →",
      n(pet_record_3("P-012")))
print("Schedule of the 21st:", n(sched_day("2026-10-21")), "tokens →",
      n(schedule_3("2026-10-21")))
print(pet_record_3("P-002")["vaccines"])

## 6. Step 4: task-shaped tools

Eleven tools that do a complete job and receive the names the staff use, not internal identifiers.

In [ ]:
def locate(pet, owner=""):
    """The pet's id from its name (and the owner's, if there are several)."""
    ids = [m for m, d in PETS.items() if strip_accents(pet) == strip_accents(d["name"])
           and strip_accents(owner) in strip_accents(OWNERS[d["owner"]]["name"])]
    if not ids:
        return None, {"error": f"There is no pet called \"{pet}\""
                               + (f" owned by {owner}" if owner else "")
                               + " at the clinic."}
    if len(ids) > 1:
        return None, {"error": f"There are {len(ids)} pets called \"{pet}\"; give "
                               "the owner's name.",
                      "pets": [pet_summary(m) for m in ids]}
    return ids[0], None


def vet_of(name):
    """The vet's id from their name or their id."""
    for v, n in VETS.items():
        if strip_accents(name) in (strip_accents(v), strip_accents(n)) or \
                strip_accents(name) in strip_accents(n):
            return v
    return None


def find_pet(name, owner=""):
    """A pet's details, looked up by name."""
    pid, error = locate(name, owner)
    return error or pet_summary(pid)


def pets_of(owner):
    """The pets of the clients whose name contains the one given."""
    ids = [d for d, x in OWNERS.items()
           if strip_accents(owner) in strip_accents(x["name"])]
    if not ids:
        return {"error": f"There is no client called \"{owner}\"."}
    return {"clients": [{"name": OWNERS[d]["name"], "pets": [
        {"name": x["name"], "species": x["species"], "breed": x["breed"]}
        for m, x in PETS.items() if x["owner"] == d]} for d in ids]}


def vaccines(pet, owner=""):
    """A pet's vaccines, each with its next date and status."""
    pid, error = locate(pet, owner)
    return error or {"pet": PETS[pid]["name"], "today": TODAY.isoformat(),
                     "vaccines": vaccines_with_status(pid)}


def history(pet, last=3, owner=""):
    """A pet's last visits."""
    pid, error = locate(pet, owner)
    if error:
        return error
    return {"pet": PETS[pid]["name"], "visits": [
        {k: v[k] for k in ("date", "reason", "treatment", "weight_kg")}
        for v in HISTORY[pid][-int(last):]]}


def dose(medicine, pet, owner=""):
    """The dose of a medicine for a pet, with allergy and species warnings."""
    med = PHARMACY.get(strip_accents(medicine))
    if med is None:
        return {"error": f"We don't have \"{medicine}\". Medicines: "
                         + ", ".join(PHARMACY) + "."}
    pid, error = locate(pet, owner)
    if error:
        return error
    m = PETS[pid]
    warnings = []
    if strip_accents(medicine) in [strip_accents(a) for a in m["allergies"]]:
        warnings.append(f"{m['name']} is allergic to {medicine}: do not prescribe.")
    if m["species"] not in med["species"]:
        warnings.append(f"{medicine} is not indicated for {m['species']}s.")
    return {"medicine": strip_accents(medicine), "pet": m["name"],
            "weight_kg": m["weight_kg"], "dose_mg_per_kg": med["dose_mg_kg"],
            "dose_mg_per_dose": round(med["dose_mg_kg"] * m["weight_kg"], 2),
            "stock": med["stock"],
            "restock": med.get("restock", "no date") if not med["stock"] else None,
            "warnings": warnings or "none"}


def free_slots(date, vet=""):
    """The free times of a day, for one vet or for all of them."""
    r = schedule_3(date)
    if "error" in r:
        return r
    vets = [vet_of(vet)] if vet else list(VETS)
    if vets == [None]:
        return {"error": f"There is no vet \"{vet}\". They are: "
                         + ", ".join(VETS.values()) + "."}
    return {"date": date, "free_times": {VETS[v]: r["free_times"][v] for v in vets}}


def appointments_of(pet, owner=""):
    """A pet's appointments, past and upcoming, with their id."""
    pid, error = locate(pet, owner)
    return error or {"pet": PETS[pid]["name"], "today": TODAY.isoformat(),
                     "appointments": [
        {"id": c["id"], "date": c["date"], "time": c["time"], "reason": c["reason"],
         "vet": VETS[c["vet"]], "status": c["status"]}
        for c in APPOINTMENTS if c["pet"] == pid]}


def accounts(owner):
    """The unpaid invoices of the clients whose name contains the one given."""
    ids = [d for d, x in OWNERS.items()
           if strip_accents(owner) in strip_accents(x["name"])]
    if not ids:
        return {"error": f"There is no client called \"{owner}\"."}
    return {"clients": [{k: v for k, v in client_record_3(d).items() if k != "id"}
                        for d in ids]}


def book(pet, vet, date, time, reason, owner=""):
    """Books an appointment from the names of the pet and the vet."""
    pid, error = locate(pet, owner)
    if error:
        return error
    v = vet_of(vet)
    if v is None:
        return {"error": f"There is no vet \"{vet}\". They are: "
                         + ", ".join(VETS.values()) + "."}
    r = book_appointment(pid, v, date, time, reason)
    if "error" in r:
        return r
    return {"booked": r["id"], "pet": PETS[pid]["name"],
            "vet": VETS[v], "date": date, "time": time}


def notify_owner_of(pet, text, owner=""):
    """Sends a notice to the owner of the pet."""
    pid, error = locate(pet, owner)
    return error or notify(PETS[pid]["owner"], text)


TASKS_SET = {"find_pet": find_pet, "pets_of": pets_of,
             "vaccines": vaccines, "history": history, "dose": dose,
             "free_slots": free_slots, "appointments_of": appointments_of,
             "accounts": accounts, "book_appointment": book,
             "cancel_appointment": cancel_appointment, "notify_owner": notify_owner_of}

In [ ]:
PET = ("Name of the pet.", True)


OWNER = ("Owner's name; only if several pets have that name.", False)


SCHEMAS_TASKS = [
    tool("find_pet", "Details of a pet looked up by its name: species, age, current "
         "weight, allergies and owner.", {"name": PET, "owner": OWNER}),
    tool("pets_of", "The pets of a client, looked up by name.",
         {"owner": ("Owner's name.", True)}),
    tool("vaccines", "Vaccines of a pet: the next date of each one and whether it is "
         "overdue.", {"pet": PET, "owner": OWNER}),
    tool("history", "The last visits of a pet: date, reason, treatment and weight.",
         {"pet": PET, "last": ("How many visits (3 by default).", False),
          "owner": OWNER}),
    tool("dose", "Dose of a medicine for a pet according to its current weight, with "
         "allergy or species warnings and the stock.",
         {"medicine": ("Name of the medicine.", True), "pet": PET, "owner": OWNER}),
    tool("free_slots", "Free times in the schedule on a day, for one vet or for all "
         "of them.", {"date": ("YYYY-MM-DD.", True),
                      "vet": ("Name of the vet; empty for all.", False)}),
    tool("appointments_of", "Appointments of a pet (past and upcoming), with their id.",
         {"pet": PET, "owner": OWNER}),
    tool("accounts", "Unpaid invoices of a client, looked up by name, and the total "
         "they owe.", {"owner": ("Owner's name.", True)}),
    tool("book_appointment", "Books an appointment at a free time.",
         {"pet": PET, "vet": ("Name of the vet.", True),
          "date": ("YYYY-MM-DD.", True),
          "time": ("HH:MM, on the hour or half past.", True),
          "reason": ("Reason for the appointment.", True), "owner": OWNER}),
    tool("cancel_appointment", "Cancels a scheduled appointment.",
         {"appointment": ("Id of the appointment, such as AP-0026 (given by "
                          "appointments_of).", True)}),
    tool("notify_owner", "Sends a notice to the owner of a pet, by email or SMS as "
         "they prefer.", {"pet": PET, "text": ("Text of the notice, in English.", True),
                          "owner": OWNER}),
]

In [ ]:
print(dose("meloxicam", "Max"))
print(dose("amoxicillin", "Simba"))
print(free_slots("2026-10-21", "Anna Sutton"))
print(find_pet("Buster"))

## 7. Twelve tasks and four toolsets

Each task is checked by the answer (dates, times, amounts, names) or by the state of the clinic: the appointments created or canceled and the notices sent.

In [ ]:
MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August",
          "September", "October", "November", "December"]


def says(answer, *forms):
    """Does any of those forms appear in the answer?"""
    return any(re.search(f, answer or "", re.I) for f in forms)


def date_in(answer, iso):
    """Does the date appear, written as 2027-08-22, 8/22/2027, August 22, 2027, Aug 22
    or 22 August?"""
    y, m, d = iso.split("-")
    month = MONTHS[int(m) - 1]
    name = rf"(?:{month}|{month[:3]}\.?)"
    return says(answer, iso, rf"\b0?{int(m)}/0?{int(d)}/{y}\b",
                rf"\b{name} 0?{int(d)}(?:st|nd|rd|th)?\b(?:,? {y})?",
                rf"\b{int(d)}(?:st|nd|rd|th)? (?:of )?{name}\b")


def time_in(answer, hhmm):
    """Does the time appear, as 16:30, 4:30 p.m., 4:30 pm or 4:30pm (4 p.m. for 16:00)?
    In a list such as "4:00, 4:30 and 5:30 p.m.", a bare 4:30 counts too."""
    h, m = map(int, hhmm.split(":"))
    h12, half = (h - 1) % 12 + 1, "p" if h >= 12 else "a"
    other = "a" if half == "p" else "p"
    forms = [rf"(?<![\d:]){h:02d}:{m:02d}(?!\d)",
             rf"(?<![\d:]){h12}:{m:02d}(?!\d)(?!\s*{other}\.?\s?m\b)"]
    if m == 0:
        forms.append(rf"(?<![\d:]){h12}\s?{half}\.?\s?m\b")
    if (h, m) == (12, 0):
        forms.append(r"\bnoon\b")
    return says(answer, *forms)


def new_appointments():
    """The appointments created during the task."""
    return [c for c in APPOINTMENTS if c["id"] not in {x["id"] for x in APPOINTMENTS0}]


def first_free(date, vet):
    """The vet's first free time on that day, in the original schedule."""
    taken = {c["time"] for c in APPOINTMENTS0 if c["date"] == date and c["vet"] == vet}
    return next(t for t in TIMES if t not in taken)


def owes(name):
    """What a client owes in unpaid invoices."""
    return round(sum(f["total"] for f in INVOICES if not f["paid"]
                     and OWNERS[f["owner"]]["name"] == name), 2)


def amount_in(answer, value):
    """Does the amount appear, as 136, 136.00, $136 or 136 dollars?"""
    whole = f"{value:.2f}".replace(".00", "")
    return says(answer, rf"\b{re.escape(whole)}(\.00?)?\b")


def notice_to(name, *forms):
    """Was a notice sent to that client, containing all those forms?"""
    return any(OWNERS[a["owner"]]["name"] == name
               and all(re.search(f, a["text"], re.I) for f in forms) for a in NOTICES)


ANY_DATE = (r"\b20\d\d-\d\d-\d\d\b", r"\b\d{1,2}/\d{1,2}/20\d\d\b",
            rf"\b(?:{'|'.join(m[:3] for m in MONTHS)})[a-z]*\.? \d{{1,2}}\b")


TASKS = [
    ("rabies_vaccine", "When is Lola, Martha Fisher's dog, due for her next rabies "
     "vaccine?",
     lambda r: date_in(r, "2027-08-22")),
    ("dose", "How many milligrams of meloxicam should Max get per dose?",
     lambda r: says(r, r"2\.07", r"2\.1\b")),
    ("allergy", "They're going to prescribe amoxicillin for Simba. Is there any "
     "problem?",
     lambda r: says(r, r"allerg")),
    ("book", "Book a checkup for Toby with Dr. Paul Meyer on Thursday the 22nd, at the "
     "first free time.",
     lambda r: [(c["pet"], c["vet"], c["date"], c["time"]) for c in new_appointments()]
     == [("P-002", "V2", "2026-10-22", first_free("2026-10-22", "V2"))]),
    ("cancel", "Cancel Rocky's appointment tomorrow.",
     lambda r: [c["id"] for c in APPOINTMENTS if c["status"] == "canceled"]
     == ["AP-0060"]),
    ("overdue_notice", "Let Toby's owner know about his overdue vaccines.",
     lambda r: notice_to("Peter Adams", "rabies", "DHPP", "leptospirosis")),
    ("debt", "How much does Oscar Price owe in unpaid invoices?",
     lambda r: amount_in(r, owes("Oscar Price"))),
    ("marthas_cat", "Martha Fisher is calling about her cat. Is any of her vaccines "
     "overdue?",
     lambda r: says(r, "FVRCP")
     and says(r, "overdue|expired|out of date|past due|lapsed|behind")),
    ("weight", "How much did Zeus weigh at his last visit, and how much has it changed "
     "since the one before?",
     lambda r: says(r, r"32\.2") and says(r, r"0\.6")
     and not says(r, r"(last|most recent) visit[^.,]{0,30}(2025|31\.6)",
                  r"31\.6 kg (at|in|during) (his|the) (last|most recent)")),
    ("last_visit", "Why did Willow come in last time, and what treatment did she get?",
     lambda r: says(r, "wound") and says(r, "amoxicillin")),
    ("free_slots", "What times is Dr. Anna Sutton free on Wednesday the 21st in the "
     "afternoon?",
     lambda r: all(time_in(r, t) for t in ("16:00", "16:30", "17:30", "18:00", "18:30"))
     and not time_in(r, "17:00")),
    ("does_not_exist", "When is Buster due for his rabies vaccine?",
     lambda r: says(r, r"\bno (pet|record|patient|animal|one|such|information)",
                    r"\bthere(?: is|'s) no\b", r"\bunable to (find|locate)",
                    r"(n't|\bnot|cannot) (exist|find|have|appear|see|locate|found|"
                    r"registered|listed|on file|in our)")
     and not says(r, *ANY_DATE)),
]

In [ ]:
def solve(functions, schemas, tasks=TASKS, max_steps=8):
    """Solves each task with the clinic reset and records the result."""
    rows = []
    for key, task, check in tasks:
        reset()
        start = time.time()
        answer, trace = agent(task, functions, schemas, max_steps)
        calls = [c for s in trace for c in s["calls"]]
        rows.append({
            "task": key, "ok": bool(check(answer)), "answer": answer,
            "steps": len(trace), "calls": [(c["name"], c["arguments"]) for c in calls],
            "errors": sum("error" in c["result"]
                          or c["result"].get("status", 200) != 200
                          for c in calls if isinstance(c["result"], dict)),
            "result_tokens": sum(c["result_tokens"] for c in calls),
            "input_tokens": sum(s["input_tokens"] for s in trace),
            "seconds": round(time.time() - start, 1), "trace": trace,
            "new_appointments": new_appointments(), "notices": NOTICES})
    return rows


def summary(rows):
    """The totals of a toolset over all the tasks."""
    return {"ok": sum(r["ok"] for r in rows),
            "finished": sum(r["answer"] is not None for r in rows),
            "steps": sum(r["steps"] for r in rows),
            "calls": sum(len(r["calls"]) for r in rows),
            "errors": sum(r["errors"] for r in rows),
            "result_tokens": sum(r["result_tokens"] for r in rows),
            "input_tokens": sum(r["input_tokens"] for r in rows),
            "seconds": round(sum(r["seconds"] for r in rows), 1)}


def toolsets():
    """The chapter's four toolsets, in the order in which they are built."""
    return {"The API as is": (API_SET, automatic(API_SET)),
            "Names and descriptions": (NAMES_SET, SCHEMAS_NAMES),
            "Results for the model": (RESULTS_SET, SCHEMAS_RESULTS),
            "Task-shaped tools": (TASKS_SET, SCHEMAS_TASKS)}

In [ ]:
results = {}
for name, (functions, schemas) in toolsets().items():
    results[name] = solve(functions, schemas)
    print(f"{name:32s}", summary(results[name]))

The tasks of each toolset, with a mark on the ones done right:

In [ ]:
for name, rows in results.items():
    print(f"{name:32s}", " ".join(("✓" if r["ok"] else "✗") + r["task"]
                                  for r in rows))

The task about Zeus's weight in steps 2, 3 and 4: what the model calls, how long each result is and what it answers.

In [ ]:
for name in list(results)[1:]:
    r = next(r for r in results[name] if r["task"] == "weight")
    print("---", name, "✓" if r["ok"] else "✗")
    for step in r["trace"]:
        for c in step["calls"]:
            print("   ", c["name"], c["arguments"], "→", c["result_tokens"],
                  "tokens")
    print("    Answer:", r["answer"])

## Exercises

1. Explain in your own words the four parts of a tool's interface. Which of them changes at each step of the chapter?

2. Print the schema that `automatic` generates for `qry` and the one written by hand for `search`. How many tokens does each take? What information does the second have that the model could not work out from the first?

3. Call by hand, without the model, `get_pat` and `pet_record_3` with Zeus's record, `P-012`, and `history` with his name. Count the tokens of each result with the tokenizer. Which fields would you still remove from the record of toolset 3?

4. Change only the description of `qry` in toolset 1 so that it says which tables exist and what the identifiers look like. How many tasks does it recover? What does that say about step 2?

5. In toolset 3, the task about Zeus's weight fails because the record only has the last visit. Make it bring the last two. Does it solve the task? Does any other task change?

6. Add to `schedule_3` an optional `vet` parameter, like the one in `free_slots`. Does toolset 3 recover the task about the free times?

7. Write three new tasks for the clinic, with their check: one that needs two tools of toolset 4, another for a client with two pets and another that asks for an impossible action. How many does the agent solve?

8. The cancel task fails in toolset 4 because the model mixes up today and tomorrow. Try two fixes: make `appointments_of` mark each appointment as "today", "tomorrow" or "past", or make `cancel_appointment` receive the pet and the date instead of the identifier. Which do you prefer, and why?

9. Merge `vaccines`, `history` and `appointments_of` into a single tool, `full_record(pet)`. Measure the success rate and the tokens. Where is the middle ground between a few big tools and many small ones?

10. **Project.** Choose an API you know (a library's, a service from your work or one you make up) and design three or four tools on top of it for a model. Write six tasks with their check, measure the API as is and your tools, and write a short report with what you changed and what improved.

## References

Model [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [Writing effective tools for agents](https://www.anthropic.com/engineering/writing-tools-for-agents) (Anthropic, 2025). Northside Veterinary Clinic and its data are fictional. Original code and experiments.